In [1]:
import pandas as pd
import joblib
import json
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [2]:
data_train = pd.read_csv('./data/features.train.csv').set_index('id')
data_train

,fnlwgt,capitalgain,capitalloss,education,age-group,hoursperweek,education-num,workclass_federal-gov,workclass_local-gov,workclass_private,...,native-country_puerto-rico,native-country_scotland,native-country_south,native-country_taiwan,native-country_thailand,native-country_trinadad&tobago,native-country_united-states,native-country_vietnam,native-country_yugoslavia,label
id,,,,,,,,,,,,,,,,,,,,,
2103,1.395884,-0.330368,6.058927,14.0,2.0,2.0,14.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
14649,-0.877088,-0.330368,-0.231477,11.0,0.0,3.0,8.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
7379,-0.594928,-0.330368,-0.231477,11.0,2.0,4.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
24479,-1.353541,-0.330368,-0.231477,15.0,2.0,3.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
19532,2.042650,-0.330368,-0.231477,3.0,3.0,2.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8695,-1.409555,-0.330368,-0.231477,15.0,0.0,2.0,9.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
2192,-0.824666,1.774050,-0.231477,15.0,2.0,2.0,9.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
8250,1.210367,-0.330368,-0.231477,11.0,0.0,2.0,8.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0


In [3]:
X = data_train.drop("label", axis=1)
y = data_train["label"]

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "weights": ["uniform", "distance"],
    "p": [1, 2]   # 1 = Manhattan, 2 = Euclidean
}

In [6]:
grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring="f1",   # ใช้ F1-score เป็นเกณฑ์
    cv=5,
    n_jobs=-1
)

In [7]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'n_neighbors': [3, 5, 7, 9, 11, 15, 21], 'p': [1, 2],
                         'weights': ['uniform', 'distance']},
             scoring='f1')

In [8]:
print("✅ Best Parameters:", grid.best_params_)
print("✅ Best F1-score (CV):", grid.best_score_)

✅ Best Parameters: {'n_neighbors': 21, 'p': 2, 'weights': 'uniform'}
✅ Best F1-score (CV): 0.7820510017965949


In [9]:
best_model = grid.best_estimator_
y_val_pred = best_model.predict(X_val)

In [10]:
print("\n📊 Validation Results:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("F1-score:", f1_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred, digits=3))


📊 Validation Results:
Accuracy: 0.81875
F1-score: 0.7972027972027973
              precision    recall  f1-score   support

           0      0.869     0.806     0.836      1653
           1      0.762     0.836     0.797      1227

    accuracy                          0.819      2880
   macro avg      0.815     0.821     0.817      2880
weighted avg      0.823     0.819     0.820      2880



In [11]:
joblib.dump(best_model,'./model/model_Knn.joblib')

['./model/model_Knn.joblib']

In [12]:
config = best_model.get_params()

In [13]:
with open('./model/config.json', 'w') as f:
    json.dump(config, f, indent=4)